# RAC Vision · clips con IA + locución (Colab, GPU T4 gratis)

1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.
2. **Entorno de ejecución → Ejecutar todas**. Tarda ~25–40 min en total (la primera vez descarga los modelos).
3. Al final se descarga `rac_resultados.zip`. Ponelo en `Desktop\rac\colab\` y avisale a Claude.

Usa **LTX-Video** (open source, Lightricks) en modo imagen→video con las imágenes reales de racvision.org, y **edge-tts** para la voz (voz argentina Tomás).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg edge-tts
import os; os.makedirs('/content/rac/in', exist_ok=True); os.makedirs('/content/rac/out', exist_ok=True)

## 1) Locución (edge-tts)

In [ ]:
import asyncio, edge_tts
LINEAS = [
  ("l1", "Bajo tierra, cada segundo cuenta.", "+4%"),
  ("l2", "RAC Vision ve lo que el ojo pierde.", "+4%"),
  ("l3", "Sin casco, sin chaleco: alerta al instante.", "+8%"),
  ("l4", "Y todo queda en tu panel, en tiempo real.", "+4%"),
  ("l5", "RAC Vision. La prevención lo cambia todo.", "+0%"),
]
VOZ = "es-AR-TomasNeural"   # alternativas: es-AR-ElenaNeural, es-MX-JorgeNeural
async def gen():
    for n, txt, rate in LINEAS:
        await edge_tts.Communicate(txt, VOZ, rate=rate).save(f"/content/rac/out/{n}.mp3")
await gen()
from IPython.display import Audio, display
for n, *_ in LINEAS: display(Audio(f"/content/rac/out/{n}.mp3"))

## 2) Imágenes de entrada (desde racvision.org)
Si alguna descarga falla, subila a mano a `/content/rac/in/` con el panel de archivos (izquierda).

In [ ]:
import urllib.request
from PIL import Image
IMGS = {"escena": "https://racvision.org/static/img3d/escena.webp",
        "retrato": "https://racvision.org/static/img3d/retrato.webp",
        "tunel": "https://racvision.org/static/img3d/tunel.webp"}
for k, u in IMGS.items():
    p = f"/content/rac/in/{k}.webp"
    try:
        urllib.request.urlretrieve(u, p); print("ok", k, Image.open(p).size)
    except Exception as e:
        print("FALTA", k, "->", e)

## 3) Clips imagen→video con LTX-Video (5 s c/u)

In [ ]:
import torch, gc
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
DT = torch.bfloat16
pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=DT)
pipe.enable_model_cpu_offload()      # entra en los 16 GB de la T4
pipe.vae.enable_tiling()
NEG = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed faces, extra limbs, warping, text, watermark"

def fit(img, w=768, h=448):
    img = img.convert("RGB"); r = max(w / img.width, h / img.height)
    img = img.resize((int(img.width * r + .5), int(img.height * r + .5)), Image.LANCZOS)
    l, t = (img.width - w) // 2, (img.height - h) // 2
    return img.crop((l, t, l + w, t + h))

CLIPS = [
  ("escena_caminan", "escena", (768, 448),
   "Three miners walk slowly toward the camera through a dark underground mine tunnel. Their orange helmet headlamps glow, warm mine lamps hang on the rock walls, dust floats in the air. The camera dollies in slowly and steadily. Cinematic, realistic, high contrast, subtle natural motion."),
  ("retrato_mira", "retrato", (448, 640),
   "A tired miner with an orange helmet and high-visibility vest slowly turns his head toward the camera inside a dark mine tunnel. His headlamp flickers softly, dust particles drift through the light. Very slow push in, shallow depth of field, cinematic, realistic."),
  ("tunel_avance", "tunel", (768, 448),
   "Slow forward camera move through an old underground mine tunnel with wooden supports and rails, caged lamps glowing warm orange, light dust in the air, steady cinematic motion, realistic."),
]
for name, src, (w, h), prompt in CLIPS:
    try:
        img = fit(Image.open(f"/content/rac/in/{src}.webp"), w, h)
    except Exception as e:
        print("salteo", name, e); continue
    g = torch.Generator("cuda").manual_seed(42)
    video = pipe(image=img, prompt=prompt, negative_prompt=NEG, width=w, height=h,
                 num_frames=121, num_inference_steps=40, guidance_scale=3.0, generator=g).frames[0]
    export_to_video(video, f"/content/rac/out/{name}.mp4", fps=24)
    print("listo", name)
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
from IPython.display import Video, display
import glob
for f in sorted(glob.glob('/content/rac/out/*.mp4')): print(f); display(Video(f, embed=True, width=480))

## 4) Descargar todo

In [ ]:
!cd /content/rac/out && zip -q -r /content/rac_resultados.zip .
from google.colab import files
files.download('/content/rac_resultados.zip')

### Opcional: Wan 2.2 (TI2V-5B) para un clip de más calidad
Más lento en T4 (~15–25 min por clip). Corré solo si querés comparar.

In [ ]:
# import torch
# from diffusers import WanImageToVideoPipeline, AutoencoderKLWan
# from diffusers.utils import export_to_video
# mid = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
# vae = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", torch_dtype=torch.float32)
# wan = WanImageToVideoPipeline.from_pretrained(mid, vae=vae, torch_dtype=torch.bfloat16)
# wan.enable_model_cpu_offload()
# img = fit(Image.open('/content/rac/in/escena.webp'), 832, 480)
# out = wan(image=img, prompt=CLIPS[0][3], negative_prompt=NEG, height=480, width=832, num_frames=81, num_inference_steps=30, guidance_scale=5.0).frames[0]
# export_to_video(out, '/content/rac/out/escena_wan.mp4', fps=16)